# Neuro-symbolic validation over extracted knowledge

A neural model can extract candidate facts from text. pyling can then apply symbolic rules to enrich those facts, flag policy violations, and create auditable explanations as RDF.

In [1]:
from pyling import reason_stream

model_output = """
@prefix : <http://example.org/ai-review#> .
:claim1 :mentions :paper42 ; :asserts :NovelMethod ; :confidence 0.91 .
:claim1 :source :llmRun17 .
:claim2 :mentions :paper42 ; :asserts :DatasetB ; :confidence 0.64 .
:claim2 :source :llmRun17 .
:NovelMethod :requiresEvidence :BenchmarkTable .
:DatasetB :requiresEvidence :DataAvailabilityStatement .
:paper42 :hasEvidence :BenchmarkTable ; :hasEvidence :DataAvailabilityStatement .
:NovelMethod :contraindicates :RetractedCitation .
:claim3 :mentions :paper42 ; :asserts :RetractedCitation ; :confidence 0.88 .
:claim3 :source :llmRun17 .
"""

rules = """
@prefix : <http://example.org/ai-review#> .
@prefix math: <http://www.w3.org/2000/10/swap/math#> .

{ ?claim :confidence ?score . ?score math:greaterThan 0.80 . } => { ?claim :reviewStatus :highConfidence } .
{ ?claim :mentions ?record ; :asserts ?item . ?item :requiresEvidence ?evidence . ?record :hasEvidence ?evidence . } => { ?claim :reviewStatus :supported } .
{ ?diagnosisClaim :mentions ?record ; :asserts ?diagnosis . ?treatmentClaim :mentions ?record ; :asserts ?treatment . ?diagnosis :contraindicates ?treatment . } => { ?treatmentClaim :reviewStatus :blockedByContraindication } .
{ ?claim :reviewStatus :highConfidence ; :reviewStatus :supported . } => { ?claim :reviewStatus :readyForHumanReview } .
"""

result = reason_stream({"sources": [rules, model_output]}, include_input_facts_in_closure=True)
print(result.closure_n3)

@prefix : <http://example.org/ai-review#> .
@prefix math: <http://www.w3.org/2000/10/swap/math#> .

:DatasetB :requiresEvidence :DataAvailabilityStatement .
:NovelMethod :contraindicates :RetractedCitation .
:NovelMethod :requiresEvidence :BenchmarkTable .
:claim1 :asserts :NovelMethod .
:claim1 :confidence 0.91 .
:claim1 :mentions :paper42 .
:claim1 :reviewStatus :highConfidence .
:claim1 :reviewStatus :readyForHumanReview .
:claim1 :reviewStatus :supported .
:claim1 :source :llmRun17 .
:claim2 :asserts :DatasetB .
:claim2 :confidence 0.64 .
:claim2 :mentions :paper42 .
:claim2 :reviewStatus :supported .
:claim2 :source :llmRun17 .
:claim3 :asserts :RetractedCitation .
:claim3 :confidence 0.88 .
:claim3 :mentions :paper42 .
:claim3 :reviewStatus :blockedByContraindication .
:claim3 :reviewStatus :highConfidence .
:claim3 :source :llmRun17 .
:paper42 :hasEvidence :BenchmarkTable .
:paper42 :hasEvidence :DataAvailabilityStatement .



In [2]:
for status in [":highConfidence", ":supported", ":blockedByContraindication", ":readyForHumanReview"]:
    print(status, result.closure_n3.count(status))

:highConfidence 2
:supported 2
:blockedByContraindication 1
:readyForHumanReview 1
